# Function / Tool Calling 基礎

## 模組脈絡：用工具呼叫把「結構」與「行動」綁定

本筆記隸屬 **03-結構收斂**。Structured Outputs 用來約束「模型回覆的資料格式」；Function / Tool Calling 則用來約束「模型要你執行哪個外部動作，以及要傳入什麼參數」。

2026 OpenAI 新專案主線使用 **Responses API tools**：

`請求(附 tools) -> response.output 出現 function_call -> 執行本地函式 -> 以 function_call_output 回填 -> 再請求得最終答案`

這個三步協定是後面 agent harness 的基礎。

## 0. 環境設定

In [ ]:
from dotenv import load_dotenv
import os
import json
from pprint import pp

from openai import OpenAI

load_dotenv()
client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. 定義本地函式與工具 schema

工具 schema 是模型能看見的 contract；真正會被執行的是你程式裡的 Python 函式。`strict=True` 搭配明確 JSON Schema，可讓工具參數更穩定。

In [ ]:
def get_current_weather(location: str, unit: str | None = "celsius") -> str:
    # 教學用假資料；production 應接真實 weather API。
    unit = unit or "celsius"
    return json.dumps(
        {
            "location": location,
            "temperature": 28,
            "unit": unit,
            "condition": "humid and cloudy",
        },
        ensure_ascii=False,
    )


tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Get current weather for a location.",
        "strict": True,
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and region, e.g. Taipei, Taiwan",
                },
                "unit": {
                    "type": ["string", "null"],
                    "enum": ["celsius", "fahrenheit", None],
                    "description": "Temperature unit.",
                },
            },
            "required": ["location", "unit"],
        },
    }
]

available_functions = {
    "get_current_weather": get_current_weather,
}

## 2. 第一次請求：讓模型決定是否呼叫工具

Responses API 的工具呼叫會出現在 `response.output`。如果 output item 的 `type` 是 `function_call`，你的程式就要讀取 `name`、`arguments`、`call_id`。

In [ ]:
input_items = [{"role": "user", "content": "今天台北市的天氣如何？請用繁體中文回答。"}]

response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_items,
    tools=tools,
)

pp(response.output)
function_calls = [item for item in response.output if item.type == "function_call"]
print("function calls:", len(function_calls))

## 3. 執行工具，並用 `function_call_output` 回填

回填時必須帶 `call_id`，讓模型知道這個工具結果對應哪一次呼叫。工具輸出通常用字串；若是結構化資料，可用 JSON 字串。

In [ ]:
for function_call in function_calls:
    function_to_call = available_functions[function_call.name]
    function_args = json.loads(function_call.arguments)
    function_result = function_to_call(**function_args)

    input_items.append(function_call)
    input_items.append(
        {
            "type": "function_call_output",
            "call_id": function_call.call_id,
            "output": function_result,
        }
    )

pp(input_items)

## 4. 第二次請求：讓模型根據工具結果產生最終答案

In [ ]:
final_response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_items,
    tools=tools,
)

print(final_response.output_text)

## 5. 問題不需要工具時，模型可以直接回答

工具只是可用能力，不代表每次都要呼叫。`tool_choice` 預設是自動判斷。

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input=[{"role": "user", "content": "寫一首關於雨天的短詩。"}],
    tools=tools,
)

pp(response.output)
print(response.output_text)

## 6. 強制呼叫特定工具

當你正在做表單填寫、參數擷取或必須經過某個工具的流程，可以用 `tool_choice` 指定工具。若只是要模型回傳結構化資料，優先用上一章的 Structured Outputs。

In [ ]:
def extract_report_metadata(company_name: str, report_date: str) -> str:
    return json.dumps(
        {"company_name": company_name, "report_date": report_date},
        ensure_ascii=False,
    )

metadata_tool = {
    "type": "function",
    "name": "extract_report_metadata",
    "description": "Extract company report metadata from user text.",
    "strict": True,
    "parameters": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "company_name": {"type": "string", "description": "報告中的公司名稱"},
            "report_date": {"type": "string", "description": "報告日期，使用 YYYY-MM-DD"},
        },
        "required": ["company_name", "report_date"],
    },
}

response = client.responses.create(
    model=OPENAI_MODEL,
    input=[{"role": "user", "content": "台積電 2023/5/1 的法說會資料摘要。"}],
    tools=[metadata_tool],
    tool_choice={"type": "function", "name": "extract_report_metadata"},
)

metadata_call = next(item for item in response.output if item.type == "function_call")
metadata = json.loads(metadata_call.arguments)
pp(metadata)

## 7. 多工具路由：用本地函式表統一執行

實務上會把工具放進 registry，由 `function_call.name` 路由到對應函式。這就是 agent executor / graph node 的雛形。

In [ ]:
def search_knowledge_base(query: str) -> str:
    # 教學用假資料；真正 RAG 會接 vector database 或 hosted file_search。
    snippets = {
        "台北天氣": "台北今日多雲偏悶熱，午後可能有短暫陣雨。",
        "Responses API": "Responses API 是 OpenAI 新專案建議入口，支援工具、狀態與多模態。",
    }
    return snippets.get(query, "沒有找到直接相關資料。")

search_tool = {
    "type": "function",
    "name": "search_knowledge_base",
    "description": "Search a tiny demo knowledge base.",
    "strict": True,
    "parameters": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "query": {"type": "string", "description": "Search query"},
        },
        "required": ["query"],
    },
}

function_registry = {
    "get_current_weather": get_current_weather,
    "search_knowledge_base": search_knowledge_base,
}

input_items = [{"role": "user", "content": "請查一下台北天氣，並用一句話提醒我是否要帶傘。"}]
response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_items,
    tools=[tools[0], search_tool],
)

for item in response.output:
    if item.type != "function_call":
        continue
    args = json.loads(item.arguments)
    result = function_registry[item.name](**args)
    input_items.extend([
        item,
        {"type": "function_call_output", "call_id": item.call_id, "output": result},
    ])

final_response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_items,
    tools=[tools[0], search_tool],
)
print(final_response.output_text)

---

## 本章小結

1. **三步協定**：附 tools 請求 -> 讀 `response.output` 的 `function_call` -> 執行本地函式 -> 用 `function_call_output` 回填 -> 再請求。
2. Tool schema 是模型看到的 contract；Python 函式才是真正執行的動作。
3. 單純要回傳型別化資料時，用 Structured Outputs；需要外部動作、查資料或改狀態時，用 function/tool calling。
4. 多工具 registry 是 agent executor 與 LangGraph node 的基礎。
5. 模型由 `OPENAI_MODEL` 控制，避免教材綁死單一模型。